In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import PIL
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.python.keras.layers import Dense, Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_dir ='/content/drive/MyDrive/output/output'
DATA_DIR = pathlib.Path(data_dir)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import tensorflow as tf

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"GPU is available: {gpu_devices}")
    # Display details about the GPU
    !nvidia-smi
else:
    print("No GPU detected. Please go to Runtime > Change runtime type and select a GPU hardware accelerator.")

tf.keras.backend.clear_session()
import gc
gc.collect()
# ==========================================
# 1. COLLECT PATHS & MAP LABELS
# ==========================================

classes = ['1', '2', '3']

file_paths = []
labels = []

for label_index, class_name in enumerate(classes):
    class_folder = os.path.join(DATA_DIR, class_name)
    if not os.path.exists(class_folder):
        continue
    for file_name in os.listdir(class_folder):
        if file_name.endswith('.npy'):
            file_paths.append(os.path.join(class_folder, file_name))
            labels.append(label_index)

print(f"Total .npy files discovered: {len(file_paths)}")

# ==========================================
# 2. PERFORM STRATIFIED SPLITS
# ==========================================
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    file_paths, labels, test_size=0.30, stratify=labels, random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=42
)

print(f"Dataset Split -> Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")


# ==========================================
# 3. DEFINE LAZY-LOADING MECHANICS (NO RESIZING)
# ==========================================

INPUT_SHAPE = (512, 512, 1)

def npy_loader_py_func(path_tensor, label_tensor):
    path_str = path_tensor.numpy().decode('utf-8')
    img_array = np.load(path_str).astype(np.float32)
    return img_array, label_tensor

def tf_parse_image(path, label):
    """Loads the array and retains its native 512x512 shape without resizing."""
    img, lbl = tf.py_function(npy_loader_py_func, [path, label], [tf.float32, tf.int32])

    img.set_shape(INPUT_SHAPE)
    lbl.set_shape(())
    return img, lbl


# ==========================================
# 4. DATA AUGMENTATION PIPELINE (512x512 COMPATIBLE)
# ==========================================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=0.15, fill_mode="reflect"),
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    layers.RandomZoom(height_factor=0.1, width_factor=0.1),
    layers.RandomBrightness(factor=0.15),
    layers.RandomContrast(factor=0.15)
], name="Random_Augmentation_Head")


# ==========================================
# 5. ASSEMBLE DATASETS
# ==========================================
def configure_tf_dataset(paths, labels, batch_size=32, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)

    # Maps the loader function without the downscaling step
    ds = ds.map(tf_parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)

    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)

    return ds.prefetch(buffer_size=tf.data.AUTOTUNE)

BATCH_SIZE = 32
train_dataset = configure_tf_dataset(train_paths, train_labels, batch_size=BATCH_SIZE, augment=True)
val_dataset   = configure_tf_dataset(val_paths, val_labels, batch_size=BATCH_SIZE, augment=False)
test_dataset  = configure_tf_dataset(test_paths, test_labels, batch_size=BATCH_SIZE, augment=False)


# ==========================================
# 6. MODEL ARCHITECTURE IN unison WITH 512x512x1
# ==========================================
def build_grayscale_resnet50_512(input_shape=(512, 512, 1), num_classes=3):
    print(f"Initializing architecture framework for input shape: {input_shape}")
    base_gray = tf.keras.applications.ResNet50(include_top=False, weights=None, input_shape=input_shape)
    base_rgb  = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', input_shape=(input_shape[0], input_shape[1], 3))

    for layer_gray, layer_rgb in zip(base_gray.layers, base_rgb.layers):
        weights_rgb = layer_rgb.get_weights()
        if not weights_rgb:
            continue
        if layer_gray.name == 'conv1_conv':
            kernel_gray = tf.reduce_mean(weights_rgb[0], axis=2, keepdims=True).numpy()
            layer_gray.set_weights([kernel_gray, weights_rgb[1]] if len(weights_rgb) > 1 else [kernel_gray])
        else:
            layer_gray.set_weights(weights_rgb)
    del base_rgb

    base_gray.trainable = True

    for layer in base_gray.layers[:-15]:
      layer.trainable = False



    # Classification Head Block
    x = base_gray.output
    gap = layers.GlobalAveragePooling2D()(x)
    gmp = layers.GlobalMaxPooling2D()(x)
    x = layers.Concatenate()([gap, gmp])
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return models.Model(inputs=base_gray.input, outputs=outputs, name='ResNet50_Grayscale_512')


model = build_grayscale_resnet50_512(input_shape=INPUT_SHAPE, num_classes=3)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ==========================================
# 7. RUN PIPELINE
# ==========================================
print("\n--- Initiating Model Training Execution (512x512) ---")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=25
)

print("\n--- Running Final Evaluation on Test Dataset ---")
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test Accuracy Result: {test_acc * 100:.2f}%")

GPU is available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Sun Jun 28 10:04:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |            